# Optional sklearn Warm-Up With Alert Scores

This optional warm-up practices a small scikit-learn pipeline on Alert lifecycle rows from canonical tiny data. It is a mechanics refresher outside the required core module sequence, not a model-quality benchmark.

In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier

from banking_fraud_lab import (
    build_foundation_progressive_views,
    evaluate_alert_scores,
    generate_minimal_banking_world,
)

## Prepare Labeled Alert Lifecycle Rows

The target comes from generated case outcomes in the Alert lifecycle, while protected scenario answer keys remain outside the learner-facing model input.

In [ ]:
tables = generate_minimal_banking_world(seed=42)
alert_lifecycle = build_foundation_progressive_views(tables)["foundation_alert_lifecycle"]

labeled_alerts = alert_lifecycle.dropna(subset=["confirmed_fraud"]).copy()
labeled_alerts["suspected_amount_chf"] = pd.to_numeric(
    labeled_alerts["suspected_amount_chf"],
    errors="raise",
)

feature_columns = ["suspected_amount_chf", "severity", "review_priority"]
target = labeled_alerts["confirmed_fraud"].astype(bool)

assert target.nunique() == 2
labeled_alerts[["alert_id", *feature_columns, "confirmed_fraud"]]

## Fit A Tiny Pipeline

A decision tree keeps the example deterministic and simple enough for a tiny-data warm-up.

In [ ]:
preprocess = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"), ["severity", "review_priority"]),
        ("numeric", "passthrough", ["suspected_amount_chf"]),
    ]
)

model = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("classifier", DecisionTreeClassifier(max_depth=1, random_state=42)),
    ]
)

model.fit(labeled_alerts[feature_columns], target)
positive_class_index = list(model.named_steps["classifier"].classes_).index(True)

alert_scores = pd.DataFrame(
    {
        "alert_id": labeled_alerts["alert_id"],
        "score": model.predict_proba(labeled_alerts[feature_columns])[:, positive_class_index],
    }
)

assert alert_scores["score"].between(0.0, 1.0).all()
alert_scores

## Evaluate Scores With Alert-Aware Metrics

Use precision, recall, alert volume, capacity, and cost summaries instead of headline accuracy.

In [ ]:
metrics = evaluate_alert_scores(
    cases=tables["cases"],
    case_outcomes=tables["case_outcomes"],
    alert_scores=alert_scores,
    thresholds=(0.25, 0.5, 0.75),
    alert_capacity=2,
)

threshold_summary = pd.DataFrame(metrics["threshold_summaries"])
assert metrics["population"]["case_count"] == len(labeled_alerts)
threshold_summary[["threshold", "precision", "recall", "alert_volume", "capacity_utilization"]]